# Linear Regression 101

Our goal is to understand the basics of running regression models for academic purposes.

---

Imports

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

## EDA

In [ ]:
# Load dataset
df = fetch_california_housing(as_frame=True).frame

# View head
df.sample(5)

To make things more interesting, we will turn column `HouseAge` into a categorical
feature.

In [ ]:
# Round to the nearest multiple of 10 and floor the result
df['HouseAge'] = (np.floor(df['HouseAge'].div(10)) * 10).astype(int).astype(str)

Let's start exploring the dataset

In [ ]:
# Print schema
df.info()

In [ ]:
# Describe numeric features
df.describe()

In [ ]:
# Describe categorical features
df.groupby('HouseAge').size()  # Can also use `df['HouseAge'].value_counts()`

Remember to count your nulls!!!

In [ ]:
# Apply a mask to the whole dataset and sum column-wise
df.isna().sum()  # Looks good!

We can also visualize the data using a histogram (only for numeric features!)

In [ ]:
# Main plot
plt.hist(x=df['MedInc'], bins=30, density=True)

# Aesthetics
plt.title('Median Income Distribution')
plt.xlabel('Median Income')
plt.ylabel('Density')

# Show
plt.show()  # plt.savefig('hist_medinc.png', dpi=2000)  # to save a fig

In order to do something similar for a non-numeric feature, we need to use a bar plot.
Do notice that the computer does not know the intrinsic order of your categories. In
fact, it may not even have one (eg. dog, cat, cow, pig).

In [ ]:
df.groupby('HouseAge').size().reset_index(name='count')

In [ ]:
[f'[{i}-{i + 10})' for i in np.arange(0, 60, step=10)]

In [ ]:
# Create temporary table where we'll store our groupby
temp = df.groupby('HouseAge').size().reset_index(name='count')

# Plot it
plt.bar(x=temp['HouseAge'], height=temp['count'])

# Aesthetics
plt.title('House Age Value Counts')
plt.xlabel('House Age (nearest 10)')
plt.ylabel('Frequency')
plt.xticks(
    ticks=range(6),
    labels=[f'[{i}-{i + 10})' for i in np.arange(0, 60, step=10)],
    rotation=45
)

# Show
plt.show()

Scatterplots are really useful to see the relationship between two features.

In [ ]:
# Main plot
plt.scatter(x=df['MedInc'], y=df['MedHouseVal'], s=4, alpha=0.1)

# Aesthetics
plt.title('Income and House Price')
plt.xlabel('Median Yearly Income (tens of thousands, USD)')
plt.ylabel('Median House Price')

# Show
plt.show()

Not a huge fan of correlation plots, but you can do that too.

In [ ]:
# Declare list with numeric columns
cols_numeric = [
    col for col in df.columns if pd.api.types.is_numeric_dtype(df.dtypes[col])
]

# Main plot
plt.matshow(A=df[cols_numeric].corr().values, cmap='RdBu_r', vmin=-1, vmax=1)

# Aesthetics
plt.colorbar(label='Correlation')  # Shows values:color
plt.title('Correlation Plot')
plt.xticks(
    ticks=range(len(cols_numeric)),
    labels=cols_numeric,
    rotation=45,
    ha='left'
)
plt.yticks(
    ticks=range(len(cols_numeric)),
    labels=cols_numeric,
    ha='right'
)

# Show
plt.show()

## Feature engineering

Creating new features is always worth it. You can add interaction terms, encode columns,
or declare any kind of feature using expert knowledge.

In [ ]:
pd.get_dummies(
    data=df[['HouseAge']],  # With column to be encoded
    prefix='HouseAge',
    dummy_na=False,
    drop_first=False,
    dtype=int
)

In [ ]:
# Add a constant term (not strictly necessary, but worth having)
df['Const'] = 1

# Turn HouseAge to binary columns (OHE) and paste that to df (without replacing)
df = pd.concat(
    objs=[
        df,  # Itself
        pd.get_dummies(
            data=df[['HouseAge']],  # With column to be encoded
            prefix='HouseAge',
            dummy_na=False,
            drop_first=False,
            dtype=int
        )
    ],
    axis=1
)

## Fitting models

Start off with a model that makes causal sense to you. Compare those results
with the least parsimonious model, and if the latter significantly improves
your results, then try trimming it down step by step (RFE maybe) to get to
a nice middle ground.

In [ ]:
# Declare categorical features separately
cols_ohe = [f'HouseAge_{i}' for i in np.arange(10, 60, step=10)]  # Set 0 as baseline
cols_m0 = ['Const', 'AveBedrms', 'AveRooms', 'MedInc'] + cols_ohe

# Baseline model
m0 = sm.OLS(
    endog=df['MedHouseVal'],
    exog=df[cols_m0],
    hasconst=True
)

# Fit
m0_res = m0.fit()

# View results
print(m0_res.summary())  # Try adding .as_latex()

Check your assumptions (such as homoskedasticity)

**NOTE:** Look back at the plots involving `MedHouseVal`. It's easy to see that the
data is censored! In other words, 5.0 is the maximum value, and values greater than
this (say, 7.8) are saved as 5.0 in the dataset. Hence, when our model predicts a
number greater than this threshold, the residual will always be $\hat{y} - 5$.

This explains the weird line in our residuals plot.


In [ ]:
# Main plot
plt.scatter(m0_res.fittedvalues, m0_res.resid, s=1, alpha=0.1)
plt.axhline(ls='--')

# Aesthetics
plt.title('Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.xlim(0, 7)

# Show
plt.show()

In [ ]:
# Main plot
plt.hist(m0_res.resid, bins=30)

# Aesthetics
plt.title('Residuals Distribution')
plt.xlabel('Residuals')
plt.ylabel('Frequency')

# Aesthetics
plt.show()

Quantile plots are super useful

In [ ]:
sm.qqplot(data=m0_res.resid.values, line='45', alpha=0.1)
plt.show()

So our error is quite simply not homoskedastic. We can change our spec to include robust
standard errors!

In [ ]:
# Mask to drop censored entries
mask = df['MedHouseVal'].lt(5)

# Model on non-censored values
m1 = sm.OLS(
    endog=df.loc[mask, 'MedHouseVal'],
    exog=df.loc[mask, cols_m0],
    hasconst=True
)

# Fit
m1_res = m1.fit()

# View results
print(m1_res.summary())

We got rid of the weird line by omitting censored values. Regardless, we still seem to
be violating the homoskedasticity assumption!

In [ ]:
# Main plot
plt.scatter(x=m1_res.fittedvalues, y=m1_res.resid, s=1, alpha=0.1)
plt.axhline(ls='--')

# Aesthetics
plt.title('Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.xlim(0, 7)

# Show
plt.show()

Let's fit a model with robust standard errors this time.

**NOTE:** Heteroskedasticity assumptions do not affect the estimated parameters. They
only affect their standard errors!!!

As a consequence, the residual plots won't change. That is, we did not overcome the
heteroskedasticity problem, but rather, we're accounting for it in the standard
errors of out estimated parameters.

In [ ]:
# Model on non-censored values
m2 = sm.OLS(
    endog=df.loc[mask, 'MedHouseVal'],
    exog=df.loc[mask, cols_m0],
    hasconst=True
)

# Fit
m2_res = m2.fit(cov_type='HC0')

# View results
print(m2_res.summary())